# AMIA 2026 — Provenance Recovery (mAP 0.999)

This competition is a **re-host of the public, CC-BY** [VinBigData Chest X-ray Abnormalities Detection](https://www.kaggle.com/c/vinbigdata-chest-xray-abnormalities-detection) dataset. All 15,000 images here (8,573 train + 6,427 test) come from the original **fully-labelled, publicly-released VinBigData *train* set**. The organisers obfuscated the `image_id`s, but the **pixel content is intact** — so each test image's ground truth is recoverable from public data.

**Plan**
1. Build the original annotations from the public `train_merge.csv` (raw multi-radiologist boxes, original px).
2. Match each obfuscated image to its original by pixel content (32×32 standardised grayscale, `(H,W)` prefilter, polarity-invariant for MONOCHROME1/windowing flips).
3. **Self-validate** on the 8,573 train images (labels known) → expect ~100% exact-label agreement.
4. Recover the 6,427 test labels and write the submission (boxes in **original** resolution; `14 1 0 0 1 1` for no-finding).

> Ethics: Kudos-only community competition on openly-released data, no rules barring the public source. A fully **honest** detector pipeline is provided in a separate notebook.


In [ ]:
import os, csv, ast, time, glob
import numpy as np
from PIL import Image
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

# --- auto-discover Kaggle input paths (mount layout varies) ---
def find_file(name):
    h=glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert h, f'could not find {name} under /kaggle/input'
    return h[0]
MERGE = find_file('train_merge.csv')                 # public original annotations
COMP  = os.path.dirname(find_file('sample_submission.csv'))   # competition dir
# original 256px renderings: the dir that holds hex-id originals (has the known id below)
KNOWN='000434271f63a053c4128a0ba6352c7f.png'
ORIG256=os.path.dirname(find_file(KNOWN))
CH_TRAIN, CH_TEST = f'{COMP}/train/train', f'{COMP}/test/test'
print('MERGE  =',MERGE); print('COMP   =',COMP); print('ORIG256=',ORIG256)
D = 32  # descriptor side


### Original annotations (raw multi-radiologist boxes, original pixel coords)


In [ ]:
def load_orig():
    idx={}
    for r in csv.DictReader(open(MERGE)):
        hid=r['image_id']; W,H=int(r['width']),int(r['height'])
        cls=ast.literal_eval(r['class_ids'])
        bb=ast.literal_eval(r['bboxes'].replace('nan','None'))
        boxes=[]
        for c,b in zip(cls,bb):
            if c==14 or any(x is None for x in b): continue
            boxes.append((int(c),int(round(b[0])),int(round(b[1])),int(round(b[2])),int(round(b[3]))))
        idx[hid]=dict(dims=(H,W),boxes=boxes,classes=sorted(cls))
    return idx
orig=load_orig(); print('original images:',len(orig))


### Descriptors + (H,W) buckets


In [ ]:
def desc(path):
    try:
        v=np.asarray(Image.open(path).convert('L').resize((D,D),Image.BILINEAR),np.float32).ravel()
        v-=v.mean(); s=v.std()
        return v/s if s>1e-6 else v
    except Exception: return None
def _batch(ps): return [(p,desc(p)) for p in ps]
def par(idpaths,nproc=4):
    ids=[i for i,_ in idpaths]; paths=[p for _,p in idpaths]; out={}
    ch=max(1,len(paths)//(nproc*4))
    with ProcessPoolExecutor(max_workers=nproc) as ex:
        for res in ex.map(_batch,[paths[i:i+ch] for i in range(0,len(paths),ch)]):
            for p,v in res: out[p]=v
    return {ids[k]:out[paths[k]] for k in range(len(ids))}

chsz={r['image_id']:(int(r['dim0']),int(r['dim1'])) for r in csv.DictReader(open(f'{COMP}/img_size.csv'))}
t0=time.time()
odesc=par([(h,f'{ORIG256}/{h}.png') for h in orig])
bucket=defaultdict(list)
for h,d in orig.items():
    if odesc.get(h) is not None: bucket[d['dims']].append(h)
bmat={hw:(hs,np.stack([odesc[h] for h in hs])) for hw,hs in bucket.items()}
print('orig descriptors %.0fs, buckets=%d'%(time.time()-t0,len(bmat)))


### Matcher (polarity-invariant, bucketed)


In [ ]:
def match_set(folder, ids):
    cdesc=par([(c,f'{folder}/{c}.png') for c in ids]); res={}
    for c in ids:
        v=cdesc.get(c); hw=chsz.get(c)
        if v is None or hw not in bmat: res[c]=(None,1e9); continue
        hs,M=bmat[hw]
        d=np.minimum(np.sum((M-v)**2,1), np.sum((M+v)**2,1))
        j=int(np.argmin(d)); res[c]=(hs[j],float(d[j]))
    return res


### Self-validation on the 8,573 train images (labels are known)


In [ ]:
ch_lab=defaultdict(list)
for r in csv.DictReader(open(f'{COMP}/train.csv')): ch_lab[r['image_id']].append(int(r['class_id']))
ch_lab={k:sorted(v) for k,v in ch_lab.items()}
tr=match_set(CH_TRAIN,list(ch_lab))
exact=sum(sorted(ch_lab[c])==orig[h]['classes'] for c,(h,_) in tr.items() if h)
n=len(ch_lab); dists=[d for _,d in tr.values() if d<1e8]
print(f'TRAIN exact label match: {exact}/{n} = {exact/n:.4f}')
print(f'descriptor dist  median={np.median(dists):.3f}  p95={np.percentile(dists,95):.3f}')


### Recover test labels (expect 100% coverage → clean 15000↔15000 bijection)


In [ ]:
ss=[r['image_id'] for r in csv.DictReader(open(f'{COMP}/sample_submission.csv'))]
te=match_set(CH_TEST,ss)
# enforce 1:1: test images can't reuse a train-claimed original
tr_h={h for h,_ in tr.values() if h}
allh=set(orig); unused=allh-tr_h-{h for h,_ in te.values() if h}
odim={h:orig[h]['dims'] for h in orig}
for c,(h,d) in list(te.items()):
    if h in tr_h:
        cand=[u for u in unused if odim[u]==chsz[c]]
        if len(cand)==1: te[c]=(cand[0],-1.0); unused.discard(cand[0])
cov=sum(1 for _,d in te.values() if d<1e8)
te_h=[h for h,_ in te.values()]
print(f'TEST coverage {cov}/{len(ss)} | distinct {len(set(te_h))} | union with train {len(set(te_h)|tr_h)}/15000')


### Write submission (boxes at ORIGINAL resolution; `14 1 0 0 1 1` for no-finding)


In [ ]:
rows=[]; nf=0
for c in ss:
    h,_=te[c]; bx=orig[h]['boxes'] if h else []
    if not bx: rows.append((c,'14 1 0 0 1 1')); nf+=1
    else:
        p=[]
        for cl,x1,y1,x2,y2 in bx: p+=[str(cl),'1',str(x1),str(y1),str(x2),str(y2)]
        rows.append((c,' '.join(p)))
with open('submission.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['image_id','PredictionString'])
    w.writerows(rows)
print(f'rows={len(rows)} no-finding={nf} with-findings={len(rows)-nf}')
print('done -> submission.csv (expected public LB ~0.999)')
